# MLP Analisys

In [ ]:
# === STEP 1: Import e configurazione Spark ===

# Standard library
import time
import json
from itertools import product

# Spark core
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, StructType, StructField

# Spark MLlib — feature engineering
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Spark MLlib — modello
from pyspark.ml.classification import MultilayerPerceptronClassifier, MultilayerPerceptronClassificationModel
# Spark MLlib — valutazione
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Spark MLlib — pipeline
from pyspark.ml import Pipeline
from pyspark.ml import PipelineModel

# Pucktrick
from pucktrick import PuckTrick, Engine

# Solo per export finale — NON usato nel loop sperimentale
import pandas as pd

In [ ]:
# === STEP 2: Configurazione Spark e caricamento dataset ===

# ── Configurazione cluster ─────────────────────────────────────────────────
MASTER_URL  = "spark://10.0.1.8:7077"
DRIVER_HOST = "10.0.1.8"

spark = SparkSession.builder \
    .appName("MetroPT_MLP_Experiments") \
    .master(MASTER_URL) \
    .config("spark.submit.deployMode",      "client") \
    .config("spark.executor.instances",     "4") \
    .config("spark.executor.cores",         "4") \
    .config("spark.executor.memory",        "13g") \
    .config("spark.driver.memory",          "8g") \
    .config("spark.driver.host",            DRIVER_HOST) \
    .config("spark.driver.bindAddress",     DRIVER_HOST) \
    .config("spark.sql.shuffle.partitions", "32") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession creata — versione:", spark.version)

# ── Costanti sperimentali ──────────────────────────────────────────────────
ALL_FEATURES = [
    "TP2_scaled", "TP3_scaled", "H1_scaled", "DV_pressure_scaled",
    "Reservoirs_scaled", "Oil_temperature_scaled", "Motor_current_scaled",
    "COMP", "DV_eletric", "Towers", "MPG", "LPS", "Caudal_impulses"
]
NOISE_FEATURES = ["DV_pressure_scaled", "Oil_temperature_scaled", "TP3_scaled"]
FEATURE_COMBINED = ["TP3_scaled", "Reservoirs_scaled"]
LABEL_COL    = "target"
NOISE_TYPES  = ["duplicated", "missing", "noisy", "outliers"]
NOISE_LEVELS = [0.1, 0.2, 0.3, 0.5]
N_RUNS       = 20
T_VALUE_95   = 2.093

# ── Caricamento dataset ────────────────────────────────────────────────────
# Nota: il file è stato salvato con pandas to_parquet (file singolo, non directory Spark).
# Si legge sul driver e si distribuisce immediatamente sul cluster via createDataFrame.
DATA_PATH = "/home/PuckTrickadmin/DATASETS/MetroDT_Modified.parquet"
raw_pdf = pd.read_parquet(DATA_PATH)
raw_df = spark.createDataFrame(raw_pdf)
del raw_pdf  # libera subito memoria driver

raw_df.printSchema()
#── Selezione colonne e cast ───────────────────────────────────────────────
df = raw_df.select(
    F.col("timestamp"),
    F.col("TP2_scaled").cast(DoubleType()),
    F.col("TP3_scaled").cast(DoubleType()),
    F.col("H1_scaled").cast(DoubleType()),
    F.col("DV_pressure_scaled").cast(DoubleType()),
    F.col("Reservoirs_scaled").cast(DoubleType()),
    F.col("Oil_temperature_scaled").cast(DoubleType()),
    F.col("Motor_current_scaled").cast(DoubleType()),
    F.col("COMP").cast(DoubleType()),
    F.col("DV_eletric").cast(DoubleType()),
    F.col("Towers").cast(DoubleType()),
    F.col("MPG").cast(DoubleType()),
    F.col("LPS").cast(DoubleType()),
    F.col("target").cast(DoubleType())
).dropna()
# ── Split temporale — coerente con il preprocessing ───────────────────────
SPLIT_DATE = "2020-06-01 00:00:00"

pt         = PuckTrick(df, engine=Engine.SPARK)
base_df    = pt.original

base_train_df = base_df.filter(F.col("timestamp") < SPLIT_DATE).drop("timestamp")
base_test_df  = base_df.filter(F.col("timestamp") >= SPLIT_DATE).drop("timestamp")

base_train_df.cache()
base_test_df.cache()

n_train       = base_train_df.count()
n_test        = base_test_df.count()
n_train_fault = base_train_df.filter(F.col("target") == 1).count()
n_test_fault  = base_test_df.filter(F.col("target") == 1).count()

print(f"Training : {n_train:,} righe ({n_train_fault:,} guasti, {n_train_fault/n_train*100:.2f}%)")
print(f"Test     : {n_test:,} righe ({n_test_fault:,} guasti, {n_test_fault/n_test*100:.2f}%)")
print(f"Split    : {n_train/(n_train+n_test)*100:.1f}% train / {n_test/(n_train+n_test)*100:.1f}% test")
print(f"\nAtteso dal preprocessing:")
print(f"  Training: ~856,832 righe, ~1.29% guasti")
print(f"  Test    : ~660,116 righe, ~2.87% guasti")

---

In [ ]:
#derivate dall'output generato
BEST_LAYERS = [13,64,2]
BEST_MAX_ITER = 100
BEST_STEP = 0.05
BEST_BLOCK = 128
BASELINE_F1 = 0.9822523417285991

Versione speciale per le casistiche di duplicated e feature combinate per noise, missing e outlier

In [ ]:
# ============================================================
# CELLA A — inject_noise aggiornata
# Modifiche rispetto alla versione precedente:
#   1. duplicated: selection_criteria = "target = 1"
#      → duplica solo la classe minoritaria (fault) per simulare overfitting
#      → affected_features non viene usato in questa branch (duplica righe intere)
#   2. missing / noisy / outliers: affected_features accetta ora anche una lista
#      → permette di passare FEATURE_COMBINED = ["TP3_scaled", "Reservoirs_scaled"]
# ============================================================

def inject_noise(pt, train_df, noise_type, feature, percentage, seed):
    """
    Inietta rumore controllato su una feature (o lista di feature) del training set.

    Args:
        pt          : istanza PuckTrick già inizializzata su base_train_df
        train_df    : DataFrame di training pulito (deve contenere ROW_ID)
        noise_type  : tipo di rumore da iniettare
        feature     : str o list — feature/features su cui applicare il rumore
        percentage  : percentuale di righe da corrompere (0.0 = nessun rumore)
        seed        : seed per riproducibilità del campionamento

    Returns:
        DataFrame con rumore iniettato, pronto per il training
    """

    # Baseline — nessun rumore
    if percentage == 0.0:
        return train_df

    # Normalizza feature in lista (supporta sia str che list)
    affected = feature if isinstance(feature, list) else [feature]

    # Strategia base condivisa
    strategy = {
        "affected_features": affected,
        "selection_criteria": "all",
        "percentage":         percentage,
        "mode":               "new",
        "perturbate_data": {
            "distribution": "random",
            "param":        {"seed": seed}
        }
    }

    if noise_type == "duplicated":
        # MODIFICA: selezioniamo solo target == 1 (classe minoritaria — guasti).
        # Duplicare solo i fault rows fa sì che il modello veda questa classe
        # più volte di quanto dovrebbe, simulando un bias da overfitting sulla
        # classe rara. affected_features non è rilevante per duplicated
        # (duplica l'intera riga), ma lo passiamo comunque per coerenza.
        dup_strategy = {
            **strategy,
            "selection_criteria": "target = 1",
        }
        status, noisy_df = pt.duplicated(train_df, dup_strategy)

    elif noise_type == "labels":
        label_strategy = {**strategy, "affected_features": [LABEL_COL]}
        status, noisy_df = pt.labels(train_df, label_strategy)

    elif noise_type == "missing":
        status, noisy_df = pt.missing(train_df, strategy)

    elif noise_type == "noisy":
        status, noisy_df = pt.noise(train_df, strategy)

    elif noise_type == "outliers":
        status, noisy_df = pt.outlier(train_df, strategy)

    else:
        raise ValueError(f"Tipo di rumore non supportato: {noise_type}")

    if status != 0:
        print(f"⚠️  inject_noise: status={status} "
              f"({noise_type}, {feature}, {percentage:.0%}, seed={seed})")
        return train_df

    return noisy_df


# ── Inizializzazione PuckTrick sul training set ────────────────────────────
pt_train = PuckTrick(base_train_df, engine=Engine.SPARK)

In [ ]:
# ============================================================
# CELLA B — pipeline e run_experiment aggiornati
# Modifiche rispetto alla versione precedente:
#   1. Aggiunto Imputer (strategia "mean") prima del VectorAssembler
#      → gestisce i NaN introdotti da missing senza far crashare MLP
#      → equivalente al handleInvalid="mean" che avevi citato
#   2. run_experiment: n_train tiene conto anche di FEATURE_COMBINED
#      (per duplicated il count cambia; per gli altri rimane N_TRAIN_BASE)
# ============================================================

from pyspark.ml.feature import Imputer

# ── Oggetti pre-costruiti — creati una volta sola ─────────────────────────

# Imputer su tutte le feature numeriche: sostituisce NaN con la media della colonna.
# Viene applicato PRIMA del VectorAssembler così il modello non riceve mai valori nulli.
base_imputer = Imputer(
    inputCols  = ALL_FEATURES,
    outputCols = ALL_FEATURES,   # sovrascrive in-place le stesse colonne
    strategy   = "mean"
)

base_assembler = VectorAssembler(
    inputCols    = ALL_FEATURES,
    outputCol    = "features",
    handleInvalid= "keep"        # fallback di sicurezza per eventuali infiniti
)

base_mlp = MultilayerPerceptronClassifier(
    featuresCol = "features",
    labelCol    = LABEL_COL,
    layers      = BEST_LAYERS,
    maxIter     = BEST_MAX_ITER,
    stepSize    = BEST_STEP,
    blockSize   = BEST_BLOCK
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol      = LABEL_COL,
    predictionCol = "prediction",
    metricName    = "f1"
)

auc_evaluator = BinaryClassificationEvaluator(
    labelCol         = LABEL_COL,
    rawPredictionCol = "rawPrediction",
    metricName       = "areaUnderROC"
)

N_TRAIN_BASE = pt_train.original.count()
print(f"✅ N_TRAIN_BASE: {N_TRAIN_BASE:,}")


def run_experiment(pt, noise_type, feature, percentage, seed):
    """
    Esegue un singolo esperimento: inietta rumore → allena MLP → valuta su test.

    feature può essere str (singola feature) o list (es. FEATURE_COMBINED).
    Il label usato nei risultati è la rappresentazione stringa di feature.
    """
    t0 = time.time()

    noisy_train = inject_noise(
        pt         = pt,
        train_df   = pt.original,
        noise_type = noise_type,
        feature    = feature,
        percentage = percentage,
        seed       = seed
    )

    mlp_run  = base_mlp.setSeed(seed * 100)

    # Pipeline: Imputer → VectorAssembler → MLP
    pipeline = Pipeline(stages=[base_imputer, base_assembler, mlp_run])
    model    = pipeline.fit(noisy_train)

    predictions = model.transform(base_test_df)
    f1  = f1_evaluator.evaluate(predictions)
    auc = auc_evaluator.evaluate(predictions)

    duration = time.time() - t0

    # Per duplicated il numero di righe cresce; per gli altri rimane fisso
    n_train = noisy_train.count() if noise_type == "duplicated" else N_TRAIN_BASE

    # Stringa leggibile per feature (supporta sia str che list)
    feature_label = "+".join(feature) if isinstance(feature, list) else feature

    return {
        "noise_type": noise_type,
        "feature":    feature_label,
        "percentage": percentage,
        "seed":       seed,
        "f1":         f1,
        "auc":        auc,
        "n_train":    n_train,
        "duration_s": duration
    }


In [ ]:
# === STEP 6b: Loop esperimenti aggiuntivi ===
# Questo step esegue due nuovi gruppi di esperimenti:
#
#   GRUPPO 1 — Duplicated con selection_criteria = "target = 1"
#              (duplica solo la classe minoritaria — fault rows)
#              feature label: "target_1"
#
#   GRUPPO 2 — Missing / Noisy / Outliers su FEATURE_COMBINED
#              (["TP3_scaled", "Reservoirs_scaled"] combinati)
#              feature label: "TP3_scaled+Reservoirs_scaled"
#
# La baseline (0%) NON viene rieseguita — già presente in mlp_results.jsonl.
# Il sistema di resume è identico al loop principale: se il processo
# si interrompe, riprende esattamente dall'ultimo run non completato.

import json
import os
from datetime import datetime

RESULTS_PATH_NEW = "/home/PuckTrickadmin/Daniel/RESULTS/mlp_results_new.jsonl"
os.makedirs(os.path.dirname(RESULTS_PATH_NEW), exist_ok=True)

# ── Label stringa per FEATURE_COMBINED (usato come chiave nel resume) ──────
COMBINED_LABEL = "+".join(FEATURE_COMBINED)   # "TP3_scaled+Reservoirs_scaled"

# ── Costruzione lista run attesi ───────────────────────────────────────────
# Struttura: (noise_type, feature_label, feature_arg, percentage, seed)
#   feature_label → stringa salvata nel JSON (per resume e analisi)
#   feature_arg   → valore passato a run_experiment (str o list)

all_runs_new = []

# GRUPPO 1 — Duplicated su target=1
for percentage in NOISE_LEVELS:
    for seed in range(1, N_RUNS + 1):
        all_runs_new.append(
            ("duplicated", "target_1", "target_1", percentage, seed)
        )

# GRUPPO 2 — Missing / Noisy / Outliers su FEATURE_COMBINED
for noise_type in ["missing", "noisy", "outliers"]:
    for percentage in NOISE_LEVELS:
        for seed in range(1, N_RUNS + 1):
            all_runs_new.append(
                (noise_type, COMBINED_LABEL, FEATURE_COMBINED, percentage, seed)
            )

total_runs_new = len(all_runs_new)
print(f"✅ Run totali previsti : {total_runs_new}")

# ── Caricamento run già completati (resume) ───────────────────────────────
completed_new = set()
if os.path.exists(RESULTS_PATH_NEW):
    with open(RESULTS_PATH_NEW, "r") as f:
        for line in f:
            try:
                r = json.loads(line)
                completed_new.add(
                    (r["noise_type"], r["feature"], r["percentage"], r["seed"])
                )
            except Exception:
                pass
    print(f"✅ Run già completati  : {len(completed_new)}")
else:
    print("✅ Nessun risultato precedente — partenza da zero")

# ── Filtra solo i run mancanti ─────────────────────────────────────────────
runs_to_do_new = [
    (nt, fl, fa, pct, seed)
    for (nt, fl, fa, pct, seed) in all_runs_new
    if (nt, fl, pct, seed) not in completed_new
]

remaining_new = len(runs_to_do_new)
print(f"✅ Run rimanenti       : {remaining_new}")
print(f"✅ Stima tempo         : {remaining_new * 60 / 3600:.1f} ore")
print(f"✅ Avvio               : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# ── Loop principale ────────────────────────────────────────────────────────
with open(RESULTS_PATH_NEW, "a") as results_file:
    for i, (noise_type, feature_label, feature_arg, percentage, seed) in enumerate(runs_to_do_new):

        try:
            result = run_experiment(
                pt         = pt_train,
                noise_type = noise_type,
                feature    = feature_arg,   # str "target_1" o list FEATURE_COMBINED
                percentage = percentage,
                seed       = seed
            )

            # Sovrascrive feature con il label stringa corretto per il salvataggio
            result["feature"]   = feature_label
            result["timestamp"] = datetime.now().isoformat()

            # Salvataggio incrementale — flush immediato per sicurezza
            results_file.write(json.dumps(result) + "\n")
            results_file.flush()

            # Log ogni 10 run
            if (i + 1) % 10 == 0:
                avg_duration = result["duration_s"]
                elapsed_h    = (i + 1) * avg_duration / 3600
                remaining_h  = (remaining_new - i - 1) * avg_duration / 3600
                print(
                    f"[{datetime.now().strftime('%H:%M:%S')}] "
                    f"Run {i+1}/{remaining_new} | "
                    f"{noise_type:<12} | {feature_label:<35} | "
                    f"{percentage:.0%} | seed={seed} | "
                    f"F1={result['f1']:.4f} | AUC={result['auc']:.4f} | "
                    f"elapsed≈{elapsed_h:.1f}h | remaining≈{remaining_h:.1f}h"
                )

        except Exception as e:
            print(f"❌ ERRORE run ({noise_type}, {feature_label}, {percentage}, seed={seed}): {e}")
            continue

print(f"\n✅ Loop completato — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"✅ Risultati salvati in: {RESULTS_PATH_NEW}")